In [1]:
import pandas as pd
from vae_module import train_vae, encode, decode, load_vae
import numpy as np

In [2]:
wti_df = pd.read_csv('../data/wti_prices.csv', index_col=0, parse_dates=['Date'])
wti_df = np.log(wti_df).diff().dropna()
cols = list(wti_df.columns)
print(wti_df.shape)
print(cols)

(480, 1)
['WTI_price']


In [3]:
predictors = pd.read_csv('../data/predictor_data.csv', index_col=0, parse_dates=['Date'])
cols = list(predictors.columns)
print(predictors.shape)
print(cols)

(481, 8)
['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD']


In [4]:
# Combine TB3MS and WTI data by month (index)
combined_df = predictors.join(wti_df, how='inner').sort_index()

# Optional: set a clear index name
combined_df.index.name = "month"

#filter data up to 2014-01-01 for training data
combined_df = combined_df[combined_df.index <= '2014-01-01']

print(combined_df.shape)
print(combined_df.isna().sum())

(336, 9)
CPI          0
TB3MS        0
M1           0
M2           0
AUD_USD      0
CAD_USD      0
NZD_USD      0
ZAR_USD      0
WTI_price    0
dtype: int64


In [5]:
from statsmodels.tsa.stattools import adfuller

# For each column we perform adf test
for column in combined_df.columns:
    result = adfuller(combined_df[column].dropna())
    print(f"ADF Statistic for {column}: {result[0]}")
    print(f"p-value for {column}: {result[1]}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value}")

# We see that for all columns except WTI price it is not stationary, so we take log difference to make it stationary, for every column except WTI price, which is already stationary
for column in combined_df.columns:
    if column != 'WTI_price':
        combined_df[column] = np.log(combined_df[column]).diff().dropna()



ADF Statistic for CPI: -0.349125611730296
p-value for CPI: 0.9182527489186103
Critical Values:
   1%: -3.4507587628808922
   5%: -2.870530068560499
   10%: -2.5715597727381647
ADF Statistic for TB3MS: -1.928922065668171
p-value for TB3MS: 0.3185799385364664
Critical Values:
   1%: -3.450632157720528
   5%: -2.870474482366864
   10%: -2.5715301325443787
ADF Statistic for M1: 3.0977607356548615
p-value for M1: 1.0
Critical Values:
   1%: -3.4503836022181056
   5%: -2.8703653471616826
   10%: -2.571471939191249
ADF Statistic for M2: 3.192138306640184
p-value for M2: 1.0
Critical Values:
   1%: -3.4510167751522642
   5%: -2.87064334231426
   10%: -2.5716201744283174
ADF Statistic for AUD_USD: -1.8801382634400816
p-value for AUD_USD: 0.3414616037894379
Critical Values:
   1%: -3.4502011472639724
   5%: -2.8702852297358983
   10%: -2.5714292194077513
ADF Statistic for CAD_USD: -1.298698392514305
p-value for CAD_USD: 0.6297675900976204
Critical Values:
   1%: -3.450081345901191
   5%: -2.8702

In [6]:
cleaned_df = combined_df.dropna()

In [11]:
from data_pipeline import build_pipeline, inverse_transform, load_scaler
import random
random.seed(4101)

# Build windowed dataset from raw training DataFrame
windows, scaler = build_pipeline(
    df=cleaned_df,            # rows = timesteps, columns = variables
    scaler_save_path="scaler.pkl"
)
# windows shape: (num_windows, window_length, num_variables)

a = windows.copy()

[Pipeline] Observations: 335 | Variables: 9
[Pipeline] Columns: ['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD', 'WTI_price']
[Pipeline] Window length: 24 | Stride: 1
[Pipeline] Windows to be created: 312
[Pipeline] Scaler: minmax
------------------------------------------------------------
[Pipeline] Normalisation complete.
[Pipeline] Data range after scaling: [0.0000, 1.0000]
[Pipeline] Windows created: 312
[Pipeline] Output shape: (312, 24, 9)  (num_windows, window_length, num_variables)
[Pipeline] Scaler saved to: scaler.pkl
------------------------------------------------------------
[Pipeline] Pipeline complete. Ready for Module 2 (VAE).


In [ ]:
# Train
from vae_module import VAEConfig

config = VAEConfig(
    latent_dim=8,
    hidden_dim=64,
    num_epochs=300,
    kl_warmup_epochs=50,
    kl_weight_max=0.005,
    learning_rate=1e-4,
    verbose_every=25,
    random_seed=42,
)
model, history = train_vae(
    data=windows,        # shape: (num_windows, window_len, num_variables)
    config=config,
    save_path="vae_checkpoint.pt"
)

# # Encode new data
# latent_sequences = encode(windows, model)

# # Decode latent sequences back to data space
# reconstructed = decode(latent_sequences, model)

# # Load a saved model
# model = load_vae("vae_checkpoint.pt")


[VAE] Device: cuda
[VAE] Input dim: 9 | Latent dim: 8 | Hidden dim: 64
[VAE] Windows: 312 | Window length: 24 | Variables: 9
[VAE] Train windows: 266 | Val windows: 46
[VAE] KL warmup: 50 epochs | Max KL weight: 0.005
------------------------------------------------------------
[VAE] Epoch    1/300 | KL weight: 0.0001 | Train — Total: 0.369476, Recon: 0.369076, KL: 4.000000 | Val   — Total: 0.378262, Recon: 0.377862, KL: 4.000000
[VAE] Epoch   25/300 | KL weight: 0.0025 | Train — Total: 0.128242, Recon: 0.118242, KL: 4.000000 | Val   — Total: 0.133561, Recon: 0.123561, KL: 4.000000
[VAE] Epoch   50/300 | KL weight: 0.0050 | Train — Total: 0.059087, Recon: 0.038985, KL: 4.020468 | Val   — Total: 0.062120, Recon: 0.041879, KL: 4.048199
[VAE] Epoch   75/300 | KL weight: 0.0050 | Train — Total: 0.046996, Recon: 0.025247, KL: 4.349847 | Val   — Total: 0.050992, Recon: 0.028891, KL: 4.420288
[VAE] Epoch  100/300 | KL weight: 0.0050 | Train — Total: 0.041608, Recon: 0.020046, KL: 4.312522 | V

In [14]:
import numpy as np

latent = encode(windows, model)
latent_flat = latent.reshape(-1, latent.shape[-1])

print("Latent mean per dimension:", latent_flat.mean(axis=0).round(4))
print("Latent std per dimension: ", latent_flat.std(axis=0).round(4))
print("Min std:", latent_flat.std(axis=0).min().round(6))

Latent mean per dimension: [ 0.5849 -0.5401 -0.6423 -0.5523 -0.642  -0.727  -0.5293  0.612 ]
Latent std per dimension:  [0.1585 0.1165 0.1527 0.1372 0.0747 0.0634 0.1325 0.0811]
Min std: 0.063435


In [24]:
# Train
from timegan_module import train_timegan, generate, load_timegan, TimeGANConfig
import torch

config = TimeGANConfig(
    hidden_dim=32,
    num_layers=2,              # increase depth slightly
    noise_dim=2,
    embedding_dim=32,
    num_epochs=2000,           # more total epochs
    batch_size=32,
    learning_rate=5e-4,        # reduce from 1e-3 — more stable
    gamma=1.0,
    phase_split=(0.1, 0.3, 0.6),  # more epochs to supervised phase
    early_stopping_patience=5,
    eval_every_n_epochs=100,   # evaluate less frequently
    verbose_every=100,
    random_seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

model, history = train_timegan(
    data=a,               # shape: (num_windows, window_length, num_variables)
    save_path="timegan_checkpoint.pt"
)

# Generate synthetic windows
synthetic = generate(n_samples=200, model=model)
# synthetic shape: (200, window_length, num_variables)

# Load saved model
model = load_timegan("timegan_checkpoint.pt")

[TimeGAN] Device: cuda
[TimeGAN] Input dim: 2 | Embedding dim: 32 | Hidden dim: 32 | Noise dim: 2
[TimeGAN] Windows: 457 | Seq length: 24
[TimeGAN] Total epochs: 1000 | Phase split — Embed: 200 | Supervised: 200 | Joint: 600
[TimeGAN] Early stopping: enabled (patience=5)

[Phase 1/3] Embedding Pretraining (200 epochs)
----------------------------------------------------------------------
  [Embedding] Epoch    1/200 | Loss: 0.113893
  [Embedding] Epoch   50/200 | Loss: 0.002172
  [Embedding] Epoch  100/200 | Loss: 0.001142
  [Embedding] Epoch  150/200 | Loss: 0.000707
  [Embedding] Epoch  200/200 | Loss: 0.000579

[Phase 2/3] Supervised Pretraining (200 epochs)
----------------------------------------------------------------------
  [Supervised] Epoch    1/200 | Loss: 0.001450
  [Supervised] Epoch   50/200 | Loss: 0.001415
  [Supervised] Epoch  100/200 | Loss: 0.001414
  [Supervised] Epoch  150/200 | Loss: 0.001421
  [Supervised] Epoch  200/200 | Loss: 0.001399

[Phase 3/3] Joint GAN T

In [25]:
import numpy as np

for v in range(a.shape[2]):
    flat = a[:, :, v].flatten()
    print(f"Variable {v}: mean={flat.mean():.4f} | std={flat.std():.4f} | "
          f"min={flat.min():.4f} | max={flat.max():.4f}")

Variable 0: mean=0.6228 | std=0.0979 | min=0.0000 | max=1.0000
Variable 1: mean=0.5129 | std=0.0847 | min=0.0000 | max=1.0000


In [27]:
for v in range(a.shape[2]):
    real_v = a[:, :, v].flatten()
    fake_v = synthetic[:, :, v].flatten()
    print(f"Variable {v}:")
    print(f"  Real  — mean: {real_v.mean():.4f} | std: {real_v.std():.4f}")
    print(f"  Synth — mean: {fake_v.mean():.4f} | std: {fake_v.std():.4f}")

Variable 0:
  Real  — mean: 0.6228 | std: 0.0979
  Synth — mean: 0.6212 | std: 0.0841
Variable 1:
  Real  — mean: 0.5129 | std: 0.0847
  Synth — mean: 0.5113 | std: 0.0304


In [28]:
real_flat = a.reshape(-1, a.shape[2])
fake_flat = synthetic.reshape(-1, synthetic.shape[2])

real_corr = np.corrcoef(real_flat.T)
fake_corr = np.corrcoef(fake_flat.T)

print("Real correlation matrix:\n", np.round(real_corr, 4))
print("Synthetic correlation matrix:\n", np.round(fake_corr, 4))
print(f"Frobenius norm: {np.linalg.norm(real_corr - fake_corr, 'fro'):.4f}")

Real correlation matrix:
 [[1.     0.1611]
 [0.1611 1.    ]]
Synthetic correlation matrix:
 [[1.     0.9753]
 [0.9753 1.    ]]
Frobenius norm: 1.1515


In [29]:
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt

n_lags = 20
synthetic = generate(len(windows), model)

fig, axes = plt.subplots(1, windows.shape[2], figsize=(6 * windows.shape[2], 4))
if windows.shape[2] == 1:
    axes = [axes]

for v in range(windows.shape[2]):
    real_series = windows[:, :, v].flatten()
    fake_series = synthetic[:, :, v].flatten()
    
    real_acf = acf(real_series, nlags=n_lags, fft=True)
    fake_acf = acf(fake_series, nlags=n_lags, fft=True)
    
    axes[v].plot(real_acf, label='Real', marker='o', markersize=4)
    axes[v].plot(fake_acf, label='Synthetic', marker='s', markersize=4)
    axes[v].axhline(0, color='black', linewidth=0.5)
    axes[v].set_title(f'ACF Comparison — Variable {v}')
    axes[v].set_xlabel('Lag')
    axes[v].legend()
    axes[v].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("acf_comparison.png", dpi=150)

In [26]:
import matplotlib.pyplot as plt
import numpy as np

synthetic = generate(len(a), model)

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
var_names = ['var_1', 'var_2']  # replace with your actual column names

for i, ax_row in enumerate(axes):
    idx = i * 30
    ax_row[0].plot(a[idx])
    ax_row[0].set_title(f'Real Window {idx}')
    ax_row[0].legend(var_names)
    
    ax_row[1].plot(synthetic[idx])
    ax_row[1].set_title(f'Synthetic Window {idx}')
    ax_row[1].legend(var_names)

plt.tight_layout()
plt.savefig("visual_overlay.png", dpi=150)

In [ ]:
# Step 1 — Module 1
windows, scaler = build_pipeline(df, scaler_save_path="scaler.pkl")

# Step 2 — Module 2 (skip for Option B, enable for hybrid)
# model_vae, _ = train_vae(windows, save_path="vae_checkpoint.pt")
# latent = encode(windows, model_vae)

# Step 3 — Module 3 (pass windows for Option B, latent for hybrid)
model_gan, history = train_timegan(windows, save_path="timegan_checkpoint.pt")

# Step 4 — Generate
synthetic_normalised = generate(500, model_gan)
synthetic_original = inverse_transform(synthetic_normalised, scaler)

In [ ]:
from evaluation_module import EvaluationConfig, evaluate_all

config = EvaluationConfig(
    n_lags=12,
    random_seed=42,
    report_save_path="results_timegan.txt",
    json_save_path="results_timegan.json",
    figure_save_path="figure_timegan.png",
)

# Run identically for each generator
results_jitter  = evaluate_all(real, synthetic_jitter,  config, variable_names=["TB3MS", "WTI_price"])
results_timegan = evaluate_all(real, synthetic_timegan, config, variable_names=["TB3MS", "WTI_price"])
results_hybrid  = evaluate_all(real, synthetic_hybrid,  config, variable_names=["TB3MS", "WTI_price"])